# mmraz-qwen-probe-evaluation-vast

Load saved Qwen MM/WMM probe artifacts from the Vast steering run, reconstruct the original training / held-out split, and evaluate probe accuracy by layer.

This notebook assumes the probes were saved by `mmraz-probe-steering-options-answer-vast.ipynb`.


In [7]:
from pathlib import Path
import hashlib
import json
import re
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 130


In [8]:
def find_data_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'data').exists():
            return p
        p = p.parent

    fallback_roots = [
        Path('/workspace/temporal-awareness'),
        Path('/workspace'),
        Path('/root/temporal-awareness'),
        Path('/root/project'),
        Path('/content'),
        Path('/content/temporal-awareness'),
    ]
    for root in fallback_roots:
        if (root / 'data').exists():
            return root

    raise RuntimeError(
        'Could not locate a data root. Put the repo or extracted data/ under /workspace/temporal-awareness, /workspace, or a similar notebook working directory.'
    )

ROOT = find_data_root(Path.cwd())
print('Data root:', ROOT)

RESULTS_ROOT_CANDIDATES = [
    ROOT / 'results' / 'vast',
    ROOT / 'results',
    Path('/workspace/temporal-awareness/results/vast'),
    Path('/workspace/results/vast'),
]

def pick_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError('None of these paths exist: ' + str(paths))

def path_for_display(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

RESULTS_ROOT = pick_first_existing(RESULTS_ROOT_CANDIDATES)
print('Results root:', RESULTS_ROOT)


Data root: /Users/michalmraz/code/spar-ai/temporal-awareness
Results root: /Users/michalmraz/code/spar-ai/temporal-awareness/results


In [9]:
STEERING_MODEL_NAME = 'Qwen/Qwen2.5-14B-Instruct'
RUN_ID = None  # set to a specific run id string like '20260328-123456' to pin artifacts

ARTIFACT_GLOB = 'mmraz_probe_steering_options_answer_vast_probe_artifacts_*.npz'
METADATA_GLOB = 'mmraz_probe_steering_options_answer_vast_probe_metadata_*.json'
PROBE_STATS_GLOB = 'mmraz_probe_steering_options_answer_vast_probe_vectors_*.csv'

def extract_run_id(path: Path) -> str | None:
    m = re.search(r'_(\d{8}-\d{6})\.[^.]+$', path.name)
    return m.group(1) if m else None

def find_latest_artifact(results_root: Path, pattern: str, run_id: str | None = None) -> Path:
    matches = sorted(results_root.glob(pattern))
    if run_id is not None:
        matches = [p for p in matches if extract_run_id(p) == run_id]
    if not matches:
        raise FileNotFoundError(f'No artifact found for pattern={pattern!r} run_id={run_id!r} under {results_root}')
    return matches[-1]

probe_artifact_path = find_latest_artifact(RESULTS_ROOT, ARTIFACT_GLOB, RUN_ID)
selected_run_id = extract_run_id(probe_artifact_path)
probe_metadata_path = find_latest_artifact(RESULTS_ROOT, METADATA_GLOB, selected_run_id)
probe_stats_path = find_latest_artifact(RESULTS_ROOT, PROBE_STATS_GLOB, selected_run_id)

print('Selected run id      :', selected_run_id)
print('Probe artifact bundle:', path_for_display(probe_artifact_path))
print('Probe metadata       :', path_for_display(probe_metadata_path))
print('Probe stats CSV      :', path_for_display(probe_stats_path))


FileNotFoundError: No artifact found for pattern='mmraz_probe_steering_options_answer_vast_probe_artifacts_*.npz' run_id=None under /Users/michalmraz/code/spar-ai/temporal-awareness/results

In [ ]:
probe_metadata = json.loads(probe_metadata_path.read_text())
probe_stats_df = pd.read_csv(probe_stats_path)
probe_bundle = np.load(probe_artifact_path)

print('Probe metadata:')
display(pd.Series(probe_metadata))
print('Probe bundle arrays:', sorted(probe_bundle.files))
display(probe_stats_df.head())


In [ ]:
def load_pairs(path: Path):
    data = json.loads(path.read_text())
    if isinstance(data, dict) and 'pairs' in data:
        return data.get('metadata', {}), data['pairs']
    return {}, data

def extract_option_letter(option_text: str) -> str | None:
    match = re.search(r'\(([ABab])\)', option_text or '')
    return match.group(1).upper() if match else None

def strip_option_label(option_text: str) -> str:
    return re.sub(r'^\s*\([ABab]\)\s*', '', option_text or '').strip()

def get_pair_option_payload(pair):
    immediate_letter = extract_option_letter(pair['immediate'])
    long_term_letter = extract_option_letter(pair['long_term'])
    if immediate_letter and long_term_letter and immediate_letter != long_term_letter:
        option_a_text = pair['immediate'] if immediate_letter == 'A' else pair['long_term']
        option_b_text = pair['immediate'] if immediate_letter == 'B' else pair['long_term']
    else:
        option_a_text = pair['immediate']
        option_b_text = pair['long_term']
    return {
        'option_a_text': option_a_text,
        'option_b_text': option_b_text,
        'candidate_immediate_text': strip_option_label(pair['immediate']),
        'candidate_long_term_text': strip_option_label(pair['long_term']),
    }

def format_binary_prompt_qwen(question: str, option_a_text: str, option_b_text: str) -> str:
    return (
        'Options:\n'
        f'{strip_option_label(option_a_text)}\n'
        f'{strip_option_label(option_b_text)}\n'
        'Answer:\n'
    )

def build_probe_training_examples_qwen(pairs, strip_option_letters=True):
    examples = []
    labels = []
    for pair in pairs:
        option_payload = get_pair_option_payload(pair)
        prompt = format_binary_prompt_qwen(
            pair['question'],
            option_payload['option_a_text'],
            option_payload['option_b_text'],
        )
        immediate_continuation = option_payload['candidate_immediate_text']
        long_term_continuation = option_payload['candidate_long_term_text']
        if not strip_option_letters:
            immediate_continuation = pair['immediate']
            long_term_continuation = pair['long_term']
        examples.append({'prompt': prompt, 'continuation': immediate_continuation, 'label': 0})
        labels.append(0)
        examples.append({'prompt': prompt, 'continuation': long_term_continuation, 'label': 1})
        labels.append(1)
    return examples, np.array(labels, dtype=np.int64)


In [ ]:
if probe_metadata.get('dataset_source', 'expanded') != 'expanded':
    raise ValueError('This evaluation notebook currently expects expanded-dataset Qwen probes.')

explicit_expanded_path = pick_first_existing([
    ROOT / 'data/raw/temporal_scope_AB_randomized/temporal_scope_explicit_expanded_500.json',
    ROOT / 'data/raw/temporal_scope/temporal_scope_explicit_expanded_500.json',
    ROOT / 'data/raw/temporal_scope_explicit_expanded_500.json',
])
implicit_expanded_path = pick_first_existing([
    ROOT / 'data/raw/temporal_scope_AB_randomized/temporal_scope_implicit_expanded_300.json',
    ROOT / 'data/raw/temporal_scope/temporal_scope_implicit_expanded_300.json',
    ROOT / 'data/raw/temporal_scope_implicit_expanded_300.json',
])

expd_meta, explicit_pairs_expd = load_pairs(explicit_expanded_path)
impd_meta, implicit_pairs_expd = load_pairs(implicit_expanded_path)

pair_indices = np.arange(len(explicit_pairs_expd))
exp_pair_train_idx, exp_pair_test_idx = train_test_split(
    pair_indices,
    test_size=0.2,
    random_state=int(probe_metadata.get('pair_split_random_state', 42)),
    shuffle=True,
)

explicit_train_pairs = [explicit_pairs_expd[i] for i in exp_pair_train_idx]
explicit_test_pairs = [explicit_pairs_expd[i] for i in exp_pair_test_idx]
implicit_pairs = list(implicit_pairs_expd)

train_examples, y_train = build_probe_training_examples_qwen(
    explicit_train_pairs,
    strip_option_letters=bool(probe_metadata.get('strip_option_letters_for_probe_training', True)),
)
test_examples, y_test = build_probe_training_examples_qwen(
    explicit_test_pairs,
    strip_option_letters=bool(probe_metadata.get('strip_option_letters_for_probe_training', True)),
)
implicit_examples, y_imp = build_probe_training_examples_qwen(
    implicit_pairs,
    strip_option_letters=bool(probe_metadata.get('strip_option_letters_for_probe_training', True)),
)

print('Expanded explicit dataset:', path_for_display(explicit_expanded_path))
print('Expanded implicit dataset:', path_for_display(implicit_expanded_path))
print('Expanded explicit metadata:', expd_meta)
print('Expanded implicit metadata:', impd_meta)
print('Pair split random_state:', probe_metadata.get('pair_split_random_state', 42))
print('Train pairs:', len(explicit_train_pairs), '| train samples:', len(y_train))
print('Test pairs :', len(explicit_test_pairs), '| test samples :', len(y_test))
print('Implicit pairs:', len(implicit_pairs), '| implicit samples:', len(y_imp))

display(pd.DataFrame([
    {'dataset': 'explicit_expanded_500', 'path': path_for_display(explicit_expanded_path), 'sha256': sha256(explicit_expanded_path)},
    {'dataset': 'implicit_expanded_300', 'path': path_for_display(implicit_expanded_path), 'sha256': sha256(implicit_expanded_path)},
]))


In [ ]:
qwen_device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Qwen device:', qwen_device)

qwen_tokenizer = AutoTokenizer.from_pretrained(STEERING_MODEL_NAME, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model_kwargs = {'trust_remote_code': True}
if qwen_device == 'cuda':
    qwen_model_kwargs['device_map'] = 'auto'
    qwen_model_kwargs['torch_dtype'] = torch.float16
elif qwen_device == 'mps':
    qwen_model_kwargs['torch_dtype'] = torch.float16
else:
    qwen_model_kwargs['torch_dtype'] = torch.float32

qwen_model = AutoModelForCausalLM.from_pretrained(STEERING_MODEL_NAME, **qwen_model_kwargs)
if qwen_device != 'cuda':
    qwen_model = qwen_model.to(qwen_device)
qwen_model.eval()
qwen_n_layers = len(qwen_model.model.layers)
print('Loaded', STEERING_MODEL_NAME, '| n_layers =', qwen_n_layers, '| hidden_size =', qwen_model.config.hidden_size)

if int(probe_bundle['layers'].shape[0]) != qwen_n_layers:
    print('Warning: artifact layer count does not match model layer count exactly.')


In [ ]:
def _move_batch_to_qwen_device(batch):
    model_device = next(qwen_model.parameters()).device
    return {key: value.to(model_device) for key, value in batch.items()}

@torch.no_grad()
def extract_mean_answer_token_activations_from_examples_qwen(examples, batch_size=1):
    activations = [[] for _ in range(qwen_n_layers)]
    model_device = next(qwen_model.parameters()).device
    autocast_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if model_device.type == 'cuda' else nullcontext()
    pad_id = qwen_tokenizer.pad_token_id if qwen_tokenizer.pad_token_id is not None else qwen_tokenizer.eos_token_id

    for start in range(0, len(examples), batch_size):
        batch_examples = examples[start:start + batch_size]
        prompt_ids_batch = []
        continuation_ids_batch = []
        seq_lengths = []
        answer_spans = []

        for example in batch_examples:
            prompt_ids = qwen_tokenizer(example['prompt'], add_special_tokens=False, return_tensors='pt')['input_ids'][0]
            continuation_ids = qwen_tokenizer(example['continuation'], add_special_tokens=False, return_tensors='pt')['input_ids'][0]
            if continuation_ids.numel() == 0:
                raise ValueError(f'Empty continuation for evaluation example: {example!r}')
            prompt_ids_batch.append(prompt_ids)
            continuation_ids_batch.append(continuation_ids)
            seq_lengths.append(int(prompt_ids.shape[0] + continuation_ids.shape[0]))

        max_seq_len = max(seq_lengths)
        input_ids = torch.full((len(batch_examples), max_seq_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch_examples), max_seq_len), dtype=torch.long)

        for row_idx, (prompt_ids, continuation_ids) in enumerate(zip(prompt_ids_batch, continuation_ids_batch)):
            seq = torch.cat([prompt_ids, continuation_ids], dim=0)
            seq_len = int(seq.shape[0])
            answer_start = int(prompt_ids.shape[0])
            answer_end = seq_len
            input_ids[row_idx, :seq_len] = seq
            attention_mask[row_idx, :seq_len] = 1
            answer_spans.append((answer_start, answer_end))

        batch = _move_batch_to_qwen_device({'input_ids': input_ids, 'attention_mask': attention_mask})
        with autocast_ctx:
            outputs = qwen_model(**batch, output_hidden_states=True, use_cache=False)

        hidden_states = outputs.hidden_states[1:]
        for layer, hidden in enumerate(hidden_states):
            pooled_rows = []
            for row_idx, (answer_start, answer_end) in enumerate(answer_spans):
                pooled = hidden[row_idx, answer_start:answer_end, :].mean(dim=0)
                pooled_rows.append(pooled.detach().float().cpu().numpy())
            activations[layer].append(np.stack(pooled_rows, axis=0))

    return [np.concatenate(parts, axis=0) for parts in activations]


In [ ]:
EVAL_BATCH_SIZE = 1

X_train = extract_mean_answer_token_activations_from_examples_qwen(train_examples, batch_size=EVAL_BATCH_SIZE)
X_test = extract_mean_answer_token_activations_from_examples_qwen(test_examples, batch_size=EVAL_BATCH_SIZE)
X_imp = extract_mean_answer_token_activations_from_examples_qwen(implicit_examples, batch_size=EVAL_BATCH_SIZE)

print('Train activation shape, layer 0   :', X_train[0].shape)
print('Held-out activation shape, layer 0:', X_test[0].shape)
print('Implicit activation shape, layer 0 :', X_imp[0].shape)


In [ ]:
artifact_layers = probe_bundle['layers'].astype(int).tolist()
mm_raw_directions = probe_bundle['mm_raw_directions']
wmm_effective_directions = probe_bundle['wmm_effective_directions']
wmm_mean_train = probe_bundle['wmm_mean_train']

rows = []
for idx, layer in enumerate(artifact_layers):
    mm_direction = mm_raw_directions[idx]
    wmm_direction = wmm_effective_directions[idx]
    mean_train = wmm_mean_train[idx]

    dataset_specs = [
        ('explicit_train', X_train[layer], y_train),
        ('explicit_test', X_test[layer], y_test),
        ('implicit_full', X_imp[layer], y_imp),
    ]
    for dataset_name, X_data, y_true in dataset_specs:
        mm_scores = X_data @ mm_direction
        wmm_scores = (X_data - mean_train) @ wmm_direction
        rows.append({
            'layer': layer,
            'dataset': dataset_name,
            'probe_variant': 'mm',
            'accuracy': float(((mm_scores > 0).astype(np.int64) == y_true).mean()),
            'score_mean': float(np.mean(mm_scores)),
            'score_std': float(np.std(mm_scores)),
        })
        rows.append({
            'layer': layer,
            'dataset': dataset_name,
            'probe_variant': 'wmm',
            'accuracy': float(((wmm_scores > 0).astype(np.int64) == y_true).mean()),
            'score_mean': float(np.mean(wmm_scores)),
            'score_std': float(np.std(wmm_scores)),
        })

eval_df = pd.DataFrame(rows).sort_values(['dataset', 'probe_variant', 'layer']).reset_index(drop=True)
display(eval_df.head(12))


In [ ]:
saved_train_cols = [c for c in probe_stats_df.columns if c in ['layer', 'mm_train_acc', 'wmm_train_acc']]
if saved_train_cols:
    saved_train_df = probe_stats_df[saved_train_cols].copy().rename(columns={
        'mm_train_acc': 'saved_mm_train_acc',
        'wmm_train_acc': 'saved_wmm_train_acc',
    })
    compare_train_df = (
        eval_df[eval_df['dataset'] == 'explicit_train']
        .pivot(index='layer', columns='probe_variant', values='accuracy')
        .reset_index()
        .rename(columns={'mm': 'recomputed_mm_train_acc', 'wmm': 'recomputed_wmm_train_acc'})
        .merge(saved_train_df, on='layer', how='left')
    )
    display(compare_train_df)
else:
    print('No saved train-accuracy columns found in probe_stats_df.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True, constrained_layout=True)
plot_order = [
    ('explicit_train', 'Explicit Train'),
    ('explicit_test', 'Explicit Held-Out Test'),
    ('implicit_full', 'Implicit Full'),
]
for ax, (dataset_name, title) in zip(axes, plot_order):
    subset = eval_df[eval_df['dataset'] == dataset_name]
    for probe_variant, color, label in [('mm', '#1f77b4', 'MM'), ('wmm', '#d62728', 'WMM')]:
        probe_df = subset[subset['probe_variant'] == probe_variant]
        ax.plot(probe_df['layer'], probe_df['accuracy'], marker='o', linewidth=1.8, color=color, label=label)
    ax.set_title(title)
    ax.set_xlabel('Layer')
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Accuracy')
axes[-1].legend(loc='lower left')
plt.show()


In [ ]:
pivot_eval = eval_df.pivot_table(index='layer', columns=['dataset', 'probe_variant'], values='accuracy')
display(pivot_eval)

save_dir = RESULTS_ROOT
save_dir.mkdir(parents=True, exist_ok=True)
eval_path = save_dir / f'mmraz_qwen_probe_evaluation_{selected_run_id}.csv'
eval_df.to_csv(eval_path, index=False)
print('Saved evaluation CSV:', path_for_display(eval_path))
